# 4 · Problem 2 — does longer context help?

BERT at 512 tokens against Longformer at 4,096. Both train on chunks, because 70%
of contracts exceed 4,096 as well. What differs is how much context one window
holds: about 300 words against about 2,600.

**Needs a GPU with real memory.** Longformer at 4,096 tokens wants a small batch
and gradient checkpointing; budget a few hours on a T4.


## Setup

Clone the repository and install. The data layer needs nothing beyond the standard
library, so this is only for the model code.


In [ ]:
!git clone -q https://github.com/ManasDasri/NNDL.git
%cd NNDL
!pip install -q -e . 'matplotlib>=3.8'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


### Prepare the corpus

Extracts answer spans from `CUAD_v1.json` and assigns document-level splits.
Chunking happens at training time, because each model needs a different window.


In [ ]:
!legal-risk-prepare --cuad_json data/CUAD_v1.json --out_dir data/processed


### Train Longformer

`[CLS]` is given global attention so the classification head reads a token that
attended to the whole window. Without it the head sees only a local neighbourhood
and the long context is wasted on the one position that matters.


In [ ]:
!legal-risk-train-transformer --model longformer --epochs 3 \
    --batch_size 1 --grad_accumulation 16 --lr 2e-5 \
    --gradient_checkpointing --output_dir outputs/longformer


### Compare against the 512-token model


In [ ]:
# Compare whatever has finished; legal-risk-compare skips missing runs.
!legal-risk-evaluate --run_dir outputs/longformer --split test --pooling max
!legal-risk-compare outputs/cnn outputs/bert outputs/legal_bert outputs/longformer


In [ ]:
print(open('docs/results.md').read())


### Cost, which belongs in the comparison

Longformer is larger and much slower. If it wins by a small margin, the honest
reporting includes what that margin cost.


In [ ]:
import json, pathlib

for name in ('bert', 'legal_bert', 'longformer'):
    path = pathlib.Path('outputs') / name / 'metrics.json'
    if not path.exists():
        print(f'{name:12s} not trained yet')
        continue
    m = json.load(open(path))
    print(f"{name:12s} {m['parameters']:>12,} params  "
          f"window {m['chunking']['window']:>5} words  "
          f"train windows {m['chunks']['train']:>6,}  "
          f"macro F1 {m['test_at_tuned_thresholds']['macro_f1']:.3f}")


### A null result is a result

Most of these clauses are locally signalled — *"shall maintain insurance"* is
recognisable from its own sentence. If Longformer does not win, that is a finding
about the task: clause detection in contracts is a local problem, and the expensive
long-context machinery is not what it needs. Say that plainly rather than treating
it as a failed experiment.
